# AI_AGG — LLM-Powered Text Aggregation

## Use Case Overview

`AI_AGG` is an aggregate function — like `SUM()` or `COUNT()` — but instead of computing a number, it synthesises a natural-language summary across many rows of text. It answers questions like "what are the common themes in these 500 comments?" without you reading each one.

**SA use case:** Summarise feedback, identify recurring issues, and generate narrative insights from grouped text data — all in a single SQL query. Perfect for executive-level reporting and automated insight generation.

**Dataset:** TPC-H `LINEITEM.L_COMMENT` — free-text notes on each shipped line item (~24M rows). We group by return flag and ship mode to answer business questions.

**What we'll demonstrate:**
1. Basic `AI_AGG` to summarise all line item comments
2. Grouped aggregation by `return_flag` (returned vs not returned)
3. Grouped aggregation by `ship_mode` to identify shipping pain points
4. Custom question-answering via `AI_AGG` prompt

In [ ]:
import os
import pandas as pd
import snowflake.connector

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME") or "SFCOGSOPS-SNOWHOUSE_AWS_US_WEST_2"
)
conn.cursor().execute("USE DATABASE GENAI_LEARNING")
conn.cursor().execute("USE SCHEMA PUBLIC")
print("Connected:", conn.account)

## Step 2 — Data Exploration

Understand the volume and structure of line item comments. We sample a small set to keep compute cost low for this demo.

In [ ]:
overview_df = pd.read_sql("""
    SELECT
        return_flag,
        ship_mode,
        COUNT(*) AS line_count,
        ROUND(SUM(extended_price),0) AS total_revenue
    FROM lineitem_comments_vw
    GROUP BY 1, 2
    ORDER BY 1, 2
""", conn)
print("Return flag distribution:")
print(overview_df.groupby("RETURN_FLAG")["LINE_COUNT"].sum().to_string())
overview_df.head(15)

## Step 3 — AI_AGG: Summarise by Return Flag

`AI_AGG(text_column, question)` reads all text values in a group and answers the question using an LLM.

We ask: *"What are the common themes in these order comments?"* for each return flag group.

> **Note:** AI_AGG processes text in batches internally. We limit to 2000 rows per group to keep latency manageable.

In [ ]:
return_flag_summary = pd.read_sql("""
    WITH sampled AS (
        SELECT return_flag, line_comment
        FROM lineitem_comments_vw
        QUALIFY ROW_NUMBER() OVER (PARTITION BY return_flag ORDER BY RANDOM()) <= 500
    )
    SELECT
        return_flag,
        AI_AGG(line_comment, 'What are the main themes and issues mentioned in these shipping comments?') AS summary
    FROM sampled
    GROUP BY return_flag
    ORDER BY return_flag
""", conn)

for _, row in return_flag_summary.iterrows():
    flag_label = {"A": "Accepted (returned & accepted)", "R": "Returned", "N": "Not returned"}.get(row["RETURN_FLAG"], row["RETURN_FLAG"])
    print(f"\n=== Return Flag: {row['RETURN_FLAG']} — {flag_label} ===")
    print(row["SUMMARY"])

## Step 4 — AI_AGG: Identify Shipping Mode Pain Points

Ask a targeted question about each shipping mode to identify operational issues.

In [ ]:
shipmode_summary = pd.read_sql("""
    WITH sampled AS (
        SELECT ship_mode, line_comment
        FROM lineitem_comments_vw
        QUALIFY ROW_NUMBER() OVER (PARTITION BY ship_mode ORDER BY RANDOM()) <= 300
    )
    SELECT
        ship_mode,
        AI_AGG(line_comment, 'Identify any recurring complaints or issues with this shipping method based on the comments.') AS issues
    FROM sampled
    GROUP BY ship_mode
    ORDER BY ship_mode
""", conn)

for _, row in shipmode_summary.iterrows():
    print(f"\n=== Ship Mode: {row['SHIP_MODE']} ===")
    print(row["ISSUES"])

## Step 5 — Interpretation & SA Tips

**What AI_AGG does internally:**
1. Collects all text values in each GROUP BY partition
2. Chunks them into batches that fit the LLM context window
3. Progressively summarises across batches
4. Returns a single coherent narrative per group

**SA tips:**
- AI_AGG is ideal for **automated narrative reports** — e.g. weekly "here's what customers said" emails generated entirely in SQL
- Pair with `RESULT_SCAN` to store summaries in a table for downstream use
- The `question` parameter is your prompt — be specific: "What percentage of comments mention delays?" works better than "summarise"
- Combine with a Task + Dynamic Table for continuous insight refresh